# Solução do Case Técnico - Previsão de Inadimplência

## 1. Visão Geral e Objetivos
Este notebook apresenta a solução para o problema de previsão de probabilidade de inadimplência de cobranças mensais. 

A solução contempla:
1. *Tratamento dos dados e união das bases* (base_cadastral, base_info e base_pagamentos).
2. *Construção da variável alvo (Target)*: Inadimplência definida como pagamento realizado com 5 ou mais dias de atraso ou não realizado.
3. *Engenharia de Features*: Criação de métricas de comprometimento financeiro, prazos contratuais e perfil do cliente.
4. *Validação Out-Of-Time (OOT)*: Separação por safras para simular o cenário real de produção sem contaminação temporal.
5. *Modelagem*: Treinamento do modelo LightGBM focado em calibração de probabilidade (Métricas: ROC-AUC e Log-Loss).

In [6]:
# Importando bibliotecas
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss
import warnings
warnings.filterwarnings('ignore')

# Configurações de exibição do Pandas
pd.set_option('display.max_columns', None)

# Carregamento robusto dos dados (autodetecta separador e remove espaços dos nomes das colunas)
df_cadastral = pd.read_csv("../data/base_cadastral.csv", sep=None, engine='python')
df_info = pd.read_csv("../data/base_info.csv", sep=None, engine='python')
df_dev = pd.read_csv("../data/base_pagamentos_desenvolvimento.csv", sep=None, engine='python')
df_test = pd.read_csv("../data/base_pagamentos_teste.csv", sep=None, engine='python')

# Limpeza automática de espaços em branco no nome das colunas
for df in [df_cadastral, df_info, df_dev, df_test]:
    df.columns = df.columns.str.strip()

print("Bases carregadas e colunas limpas com sucesso!")

Bases carregadas e colunas limpas com sucesso!


## 2. Construção da Variável Target
A variável target é definida com base na diferença em dias entre a DATA_PAGAMENTO e a DATA_VENCIMENTO:
- *TARGET = 1* (Inadimplente): Pagamento com atraso $\ge 5$ dias ou cobrança não paga (DATA_PAGAMENTO nula).
- *TARGET = 0* (Adimplente): Pagamento realizado com menos de 5 dias de atraso.

In [7]:
# Conversão de colunas de data
df_cadastral["DATA_CADASTRO"] = pd.to_datetime(df_cadastral["DATA_CADASTRO"])

for df in [df_dev, df_test]:
    df["DATA_EMISSAO_DOCUMENTO"] = pd.to_datetime(df["DATA_EMISSAO_DOCUMENTO"])
    df["DATA_VENCIMENTO"] = pd.to_datetime(df["DATA_VENCIMENTO"])
    df["SAFRA_REF_DT"] = pd.to_datetime(df["SAFRA_REF"])

df_dev["DATA_PAGAMENTO"] = pd.to_datetime(df_dev["DATA_PAGAMENTO"])

# Cálculo dos dias de atraso e criação da variável alvo
dias_atraso = (df_dev["DATA_PAGAMENTO"] - df_dev["DATA_VENCIMENTO"]).dt.days
df_dev["TARGET"] = np.where((dias_atraso >= 5) | (df_dev["DATA_PAGAMENTO"].isna()), 1, 0)

print("Distribuição da variável Target:")
print(df_dev["TARGET"].value_counts(normalize=True))

Distribuição da variável Target:
TARGET
0    0.92978
1    0.07022
Name: proportion, dtype: float64


## 3. Engenharia de Features e Merge das Bases
Para enriquecer a capacidade do modelo, relacionamos as 3 tabelas e criamos variáveis de negócio:
- *RATIO_VALOR_RENDA*: Nível de comprometimento da parcela em relação à renda declarada no mês anterior.
- *RENDA_POR_FUNC*: Capacidade financeira por funcionário da empresa.
- *PRAZO_PAGAMENTO_DIAS*: Janela concedida entre emissão e vencimento.
- *TEMPO_RELACIONAMENTO_DIAS*: Tempo de cadastro do cliente no sistema até a safra atual.

In [8]:
def construir_features(df_pagamentos, df_cad, df_inf):
    # Relacionamento entre tabelas (ID_CLIENTE e SAFRA_REF)
    df = df_pagamentos.merge(df_cad, on="ID_CLIENTE", how="left")
    df = df.merge(df_inf, on=["ID_CLIENTE", "SAFRA_REF"], how="left")
    
    # Preenchimento de nulos em variáveis numéricas
    df["RENDA_MES_ANTERIOR"] = df["RENDA_MES_ANTERIOR"].fillna(0)
    df["NO_FUNCIONARIOS"] = df["NO_FUNCIONARIOS"].fillna(0)
    
    # Indicadores financeiros e proporções
    df["RATIO_VALOR_RENDA"] = df["VALOR_A_PAGAR"] / (df["RENDA_MES_ANTERIOR"] + 1)
    df["RENDA_POR_FUNC"] = df["RENDA_MES_ANTERIOR"] / (df["NO_FUNCIONARIOS"] + 1)
    
    # Features temporais
    df["PRAZO_PAGAMENTO_DIAS"] = (df["DATA_VENCIMENTO"] - df["DATA_EMISSAO_DOCUMENTO"]).dt.days
    df["TEMPO_RELACIONAMENTO_DIAS"] = (df["SAFRA_REF_DT"] - df["DATA_CADASTRO"]).dt.days
    
    # Mapeamento do tipo de cliente (PF/PJ)
    df["FLAG_PF_NUM"] = df["FLAG_PF"].apply(lambda x: 1 if str(x).upper() == "X" else 0)
    
    # Tratamento de variáveis categóricas
    cat_cols = ["DDD", "SEGMENTO_INDUSTRIAL", "DOMINIO_EMAIL", "PORTE", "CEP_2_DIG"]
    for c in cat_cols:
        df[c] = df[c].astype("category")
        
    return df

df_dev_feat = construir_features(df_dev, df_cadastral, df_info)
df_test_feat = construir_features(df_test, df_cadastral, df_info)

features = [
    "VALOR_A_PAGAR", "TAXA", "RENDA_MES_ANTERIOR", "NO_FUNCIONARIOS",
    "RATIO_VALOR_RENDA", "RENDA_POR_FUNC", "PRAZO_PAGAMENTO_DIAS",
    "TEMPO_RELACIONAMENTO_DIAS", "FLAG_PF_NUM", "DDD",
    "SEGMENTO_INDUSTRIAL", "DOMINIO_EMAIL", "PORTE", "CEP_2_DIG"
]

print("Features construídas com sucesso!")

Features construídas com sucesso!


## 4. Validação Out-Of-Time (OOT)
Para garantir que o modelo generalize para safras futuras sem vazamento de dados, realizamos a validação reservando a *última safra disponível* na base de desenvolvimento como conjunto de validação temporal.

In [9]:
safras = sorted(df_dev_feat["SAFRA_REF"].unique())
safra_validacao = safras[-1]

df_train = df_dev_feat[df_dev_feat["SAFRA_REF"] != safra_validacao]
df_val = df_dev_feat[df_dev_feat["SAFRA_REF"] == safra_validacao]

X_tr, y_tr = df_train[features], df_train["TARGET"]
X_va, y_va = df_val[features], df_val["TARGET"]

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

preds_val = model.predict_proba(X_va)[:, 1]
print(f"Safra de Validação: {safra_validacao}")
print(f"ROC-AUC em Validação OOT: {roc_auc_score(y_va, preds_val):.4f}")
print(f"Log-Loss em Validação OOT: {log_loss(y_va, preds_val):.4f}")

[LightGBM] [Info] Number of positive: 5291, number of negative: 69610
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000530 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1757
[LightGBM] [Info] Number of data points in the train set: 74901, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.070640 -> initscore=-2.576901
[LightGBM] [Info] Start training from score -2.576901
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

## 5. Retreino Final e Geração da Submissão
Treinamos o modelo final utilizando 100% dos dados históricos de desenvolvimento e realizamos a predição das probabilidades de inadimplência para a base base_pagamentos_teste.csv.

In [10]:
# Retreino final com todo o conjunto de desenvolvimento
final_model = lgb.LGBMClassifier(
    n_estimators=model.best_iteration_ or 300,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
final_model.fit(df_dev_feat[features], df_dev_feat["TARGET"])

# Predição no teste
df_test_feat["PROBABILIDADE_INADIMPLENCIA"] = final_model.predict_proba(df_test_feat[features])[:, 1]

# Exportação do arquivo final no formato exato solicitado no desafio
submissao = df_test_feat[["ID_CLIENTE", "SAFRA_REF", "PROBABILIDADE_INADIMPLENCIA"]]
submissao.to_csv("submissao_case.csv", index=False)

print("Arquivo 'submissao_case.csv' gerado com sucesso!")
print(submissao.head())

[LightGBM] [Info] Number of positive: 5436, number of negative: 71978
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000575 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1761
[LightGBM] [Info] Number of data points in the train set: 77414, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.070220 -> initscore=-2.583317
[LightGBM] [Info] Start training from score -2.583317
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain